In [1]:
from pipelines.readers.b3_indices_segmentos_setoriais import ReaderSQLB3IndicesSegmentosSetoriais
from pipelines.readers.cvm_balanco_patrimonial import ReaderCSVCVMBalancoPatrimonial

import plotly.express as px

In [10]:
reader_b3_segmentos_setoriais = ReaderSQLB3IndicesSegmentosSetoriais()
df_b3_segmentos_setoriais = reader_b3_segmentos_setoriais.read(indice="IBEP")

print(df_b3_segmentos_setoriais.shape)
df_b3_segmentos_setoriais.tail(3)

(71, 7)


,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%)
68,IBEP,2026-07-06,VIVT3,TELEF BRASIL,ON EJ,707125712,1.166
69,IBEP,2026-07-06,WEGE3,WEG,ON NM,1485954732,3.278
70,IBEP,2026-07-06,YDUQ3,YDUQS PART,ON NM,261365845,0.112


In [22]:
reader_cvm_balanco_patrimonial = ReaderCSVCVMBalancoPatrimonial()

tickers_erros = []

infos = {}

for codigo in df_b3_segmentos_setoriais["Código"]:
    
    try:
        
        # arquivo: DRE_ind | codigo_conta: 3.03 | info: Resultado Bruto
        # arquivo: DRE_ind | codigo_conta: 3.01 | info: Receita de Venda de Bens e/ou Serviços
        # arquivo: BPA_ind | codigo_conta: 1    | info: Ativo Total

        df_cvm_balanco_patrimonial = reader_cvm_balanco_patrimonial.read(ticker=codigo, tipo_arquivo="BPA_ind", cd_conta="1")
    
    except Exception as e:
        
        tickers_erros.append(codigo)
        
        continue
    
    name = df_cvm_balanco_patrimonial.columns[-1]
    value = df_cvm_balanco_patrimonial.iloc[:, 1].ffill().pct_change(1).iloc[-1] * 100
    
    infos[codigo] = {"name": name, "value": value}
    
print("Tickers com erro:", tickers_erros)

Tickers com erro: ['MOTV3']


In [23]:


nome_map = {
    ticker: dados["name"]
    for ticker, dados in infos.items()
}

valor_map = {
    ticker: dados["value"]
    for ticker, dados in infos.items()
}

df = df_b3_segmentos_setoriais.copy()

df["nome"] = df["Código"].map(nome_map)
df["value"] = df["Código"].map(valor_map)


df.tail(3)

,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%),nome,value
68,IBEP,2026-07-06,VIVT3,TELEF BRASIL,ON EJ,707125712,1.166,TELEFÔNICA_BRASIL_S.A_Ativo_Total,1.933082
69,IBEP,2026-07-06,WEGE3,WEG,ON NM,1485954732,3.278,WEG_S.A._Ativo_Total,3.281171
70,IBEP,2026-07-06,YDUQ3,YDUQS PART,ON NM,261365845,0.112,YDUQS_PARTICIPACOES_S.A._Ativo_Total,-6.505926


In [24]:
plot_df = (
    df[["Código", "value"]]
    .dropna()
    .sort_values("value")
)

fig = px.bar(
    plot_df,
    x="value",
    y="Código",
    orientation="h",
    color="value",
    color_continuous_scale="RdYlGn",
    labels={"value": "Variação (%)", "Código": "Empresa"},
    title="Variação do Patrimônio por Empresa"
)

fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(height=1000, width=1200, yaxis={"categoryorder": "total ascending"})
fig.update_yaxes(tickmode="array", tickvals=plot_df["Código"], ticktext=plot_df["Código"])
fig.show()


In [14]:
df[df.isna().any(axis=1)]

,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%),nome,value
42,IBEP,2026-07-06,MOTV3,MOTIVA SA,ON NM,1002196725,0.705,NaN,NaN


In [25]:
# select a specific ticker to analyze
ticker_select = "TOTS3"

In [26]:
df[df["Código"] == ticker_select]

,Indice,Data Referencia,Código,Ação,Tipo,Qtde. Teórica,Part. (%),nome,value
61,IBEP,2026-07-06,TOTS3,TOTVS,ON NM,533997411,0.76,TOTVS_S.A._Ativo_Total,36.947557


In [27]:
print(f"Media da variação do patrimônio das empresas do índice: {df['value'].mean():.2f}%")

Media da variação do patrimônio das empresas do índice: 1.42%


In [28]:
print(f"Diferenca absoluta: {df[df['Código'] == ticker_select]['value'].values[0] - df['value'].mean():.2f}")

Diferenca absoluta: 35.53
